[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH06/06_02_begin/notebooks/06_02_llm_judge.ipynb)

# 06_02 Evaluate with an LLM judge

Use a frontier model to score replies on qualities a rule checker can't measure, like grounding and how clear the next step is.

This notebook doesn't need a GPU. The default CPU runtime works fine. It does need an Anthropic API key. You don't need one for this course: watch this demo, then build your own judge in the next AI Practice.

In [ ]:
%%capture
!pip install anthropic

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH06")

import ridgeline

In [ ]:
import os
from google.colab import userdata

# Add your key in Colab Secrets as ANTHROPIC_API_KEY, with notebook access turned on.
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()
JUDGE_MODEL = "claude-sonnet-5-5"

## Load the replies to judge

In [ ]:
# Outputs from 06_01. Use the course copy, or change the path to your own file.
outputs = {o["id"]: o for o in ridgeline.load_tickets(ridgeline.DATA / "eval_outputs.jsonl")}
tickets = {t["id"]: t for t in ridgeline.load_tickets()}
print(f"Judging the {len(tickets)} hand-written seed tickets")

## Write the rubric

In [ ]:
# TODO: Write JUDGE_PROMPT with the four dimensions, a 1 to 5 scale, the voice rules, the policies, the ticket, and the reply. Ask for a JSON line of scores at the end.


## Build the judge

In [ ]:
# TODO: Write judge(ticket, reply) that sends the prompt to Claude and returns the four scores as integers.
# Claude's reply can start with a thinking block, so read only the blocks where block.type == "text".


## Judge both models

In [ ]:
rows = []
for ticket_id, ticket in tickets.items():
    for model_name in ["baseline", "finetuned"]:
        scores = judge(ticket, outputs[ticket_id][model_name])
        rows.append({"id": ticket_id, "model": model_name, **scores})
    print(f"  judged {ticket_id}")

## Compare the averages

In [ ]:
import pandas as pd

scores = pd.DataFrame(rows)
scores["average"] = scores[DIMENSIONS].mean(axis=1)
scores["passes"] = (scores["average"] >= 4) & (scores[DIMENSIONS].min(axis=1) >= 3)
scores.groupby("model")[DIMENSIONS + ["average", "passes"]].mean().round(2)

## Read the lowest-scoring reply

In [ ]:
worst = scores[scores["model"] == "finetuned"].sort_values("average").iloc[0]
print(f"Lowest-scoring fine-tuned reply: {worst['id']}")
print(worst[DIMENSIONS].to_dict(), "\n")
print(outputs[worst["id"]]["finetuned"])